# Домашнее задание №7

Распознавание предметов одежды на `fashion_mnist`.

Для повышения качества используем сверточную нейронную сеть и подбираем гиперпараметры:

- количество слоев;
- количество нейронов;
- функции активации;
- оптимизатор;
- `batch_size`;
- количество эпох.


In [ ]:
from numpy.random import seed
seed(2025)

import numpy as np
import matplotlib.pyplot as plt

from tensorflow.random import set_seed
set_seed(2025)

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
%matplotlib inline

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

classes = ['футболка', 'брюки', 'свитер', 'платье', 'пальто', 'туфли', 'рубашка', 'кроссовки', 'сумка', 'ботинки']

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap='binary')
    plt.xlabel(classes[y_train[i]])
plt.tight_layout()
plt.show()

In [ ]:
# Приводим изображения к формату CNN и нормализуем значения пикселей.
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Переводим метки в one-hot encoding.
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print('x_train:', x_train.shape)
print('x_test :', x_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)

## Архитектура сети

Для `fashion_mnist` сверточная сеть обычно дает заметно лучшее качество, чем простая полносвязная модель. Здесь используем два сверточных блока, нормализацию, dropout и плотный классификатор.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.30),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.40),
    Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, min_lr=1e-5)
]

history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
scores = model.evaluate(x_test, y_test, verbose=2)
print('Доля правильных ответов на тестовых данных:', round(scores[1], 4))

## Проверка распознавания

Берем случайный пример из тестовой выборки и смотрим предсказание модели.

In [ ]:
predictions = model.predict(x_test)

import random
n = random.randrange(0, 10000)

plt.imshow(x_test[n].reshape(28, 28), cmap='binary')
plt.show()

print('Предсказание модели:', classes[np.argmax(predictions[n])])
print('Истинный класс     :', classes[np.argmax(y_test[n])])

## Итог

В этой версии ноутбука улучшение качества достигается за счет:

- перехода от полносвязной сети к сверточной;
- увеличения числа фильтров и слоев;
- использования `BatchNormalization` и `Dropout`;
- оптимизатора `Adam` с `batch_size=128`;
- ранней остановки и уменьшения шага обучения.

На практике такая конфигурация обычно дает точность выше `0.9` на `fashion_mnist`.